# Clase 19 asíncrona: ordenamiento topológico y Kahn

## Pregunta inicial

**¿Cómo encontramos un orden válido para ejecutar tareas cuando unas dependen de otras?**

<div style="border:1px solid #9bb8d3;border-left:6px solid #315f8c;background:#eef5fb;color:#17212b;padding:14px 18px;margin:14px 0;border-radius:4px;line-height:1.55"><strong style="color:#0f2f4f">INSTRUCCIÓN — Cuaderno de trabajo</strong><br>Predice cada actualización, completa las actividades y responde en <code style="color:#102a43;background:#dbeaf5;padding:2px 5px;border-radius:3px;font-weight:600">notebook.md</code>. No respondas dentro de este archivo.</div>

**Responde esta pregunta en notebook.md.**


## Objetivos de aprendizaje

Al terminar podrás:

- interpretar aristas como precedencias y reconocer DAG;
- calcular grados de entrada y nodos disponibles;
- ejecutar e implementar Kahn con una cola;
- explicar invariantes y detectar ciclos por cobertura;
- normalizar vecinos implícitos, aislados y duplicados sin mutar;
- validar órdenes cuando existen varias respuestas;
- modelar cursos y adaptar índices de CSES;
- comparar Kahn con BFS y con una extensión basada en heap;
- justificar O(V+E) y diseñar pruebas por propiedades.

### Ruta sugerida para cuatro horas

1. **Modelar dependencias** — secciones 1 a 7.
2. **Descubrir y trazar Kahn** — secciones 8 a 11.
3. **Implementar, detectar y validar** — secciones 12 a 18.
4. **Aplicar, probar e integrar** — secciones 19 a 25.


## 1. Presentación de la clase

Esta ruta asíncrona dura aproximadamente cuatro horas. Imagina que debes construir un plan de estudios: algunos cursos pueden comenzar hoy, mientras otros esperan uno o varios prerrequisitos. El objetivo no es encontrar un camino entre dos cursos; necesitamos una secuencia que respete **todas** las restricciones.

La narrativa del cierre continúa: BFS procesa descubrimientos con una cola; 0-1 BFS usa una deque; Dijkstra extrae la distancia mínima con un heap; Kruskal consulta componentes con Union-Find. Ahora la operación repetida será **elegir una tarea sin requisitos pendientes**. La representaremos con una cola y grados de entrada.

```text
dependencia dirigida → requisitos pendientes → grado cero → cola → orden
```

<div style="border-left:6px solid #315f8c;background:#eef5fb;color:#17212b;padding:14px 18px;margin:14px 0"><strong>RUTA ASÍNCRONA</strong><br>Lee, predice, completa trazas, usa el visualizador, implementa y prueba. Conserva tus respuestas en <code>notebook.md</code>.</div>

Al terminar esta sección debe quedar clara la pregunta, no todavía la receta. La siguiente sección hará explícita la dirección de cada dependencia.

### Comprueba tu comprensión

**Pregunta:** En el problema de cursos, ¿qué operación necesitamos repetir para construir un plan válido?

**Responde esta pregunta en notebook.md.**


## 2. El nuevo problema: dependencias

Nuestro problema conductor es este plan:

```text
Programación → Estructuras de Datos
Matemáticas Discretas → Estructuras de Datos
Estructuras de Datos → Algoritmos
Álgebra → Optimización
Algoritmos → Proyecto Final
Optimización → Proyecto Final
```

Programación, Matemáticas Discretas y Álgebra pueden comenzar sin esperar. Estructuras de Datos necesita dos requisitos; Proyecto Final espera dos ramas completas. El trabajo repetido es decidir qué está disponible **ahora**, registrar lo realizado y actualizar lo que deja de estar bloqueado.

Situaciones distintas comparten la misma operación: una compilación espera archivos previos, un pipeline espera datos y una receta espera preparaciones. No son ejemplos aislados: en todos, una tarea se habilita cuando desaparece su último requisito.

**Actividad inicial.** Anota qué cursos tomarías primero, cuáles esperan y si crees que existe un solo plan. Resultado comprobable: cualquier propuesta debe colocar cada prerrequisito antes que su curso.

<div style="border-left:6px solid #b26a00;background:#fff6e6;color:#3d2b10;padding:14px 18px;margin:14px 0"><strong>ADVERTENCIA</strong><br>Elegir el curso más corto, más fácil o escrito primero no protege las dependencias.</div>

Ya identificamos la necesidad; ahora fijaremos qué significa la flecha.

### Comprueba tu comprensión

**Pregunta:** ¿Qué debe ocurrir antes de Proyecto Final y por qué puede haber más de una respuesta correcta?

**Responde esta pregunta en notebook.md.**


## 3. Interpretación de aristas dirigidas

En un grafo de dependencias, `A → B` significa **A debe completarse antes que B**. No expresa una relación simétrica. Invertir la flecha cambia el contrato: `B → A` obligaría a terminar B primero.

```python
grafo = {
    "Programación": ["Estructuras de Datos"],
    "Estructuras de Datos": ["Algoritmos"],
    "Algoritmos": ["Proyecto Final"],
    "Proyecto Final": [],
}
```

Cada clave enumera sus **sucesores**, no sus prerrequisitos. En compilación, `parser.c → programa` indica que el programa espera al objeto generado; en una receta, `precalentar → hornear`; en un pipeline, `limpiar → entrenar`.

**Comprobación manual.** Para `A → C` y `B → C`, C debe esperar dos tareas. A y B no están ordenadas entre sí. Dibuja también las flechas inversas y describe cómo cambia la historia.

<div style="border-left:6px solid #b42318;background:#fff0ef;color:#341313;padding:14px 18px;margin:14px 0"><strong>PRECAUCIÓN</strong><br>Muchos jueces usan pares con otra convención. En esta clase <code>(prerrequisito, curso)</code> significa prerrequisito → curso.</div>

Con la dirección fijada, podemos distinguir dependencias compatibles de dependencias imposibles.

### Comprueba tu comprensión

**Pregunta:** ¿Por qué A → B y B → A representan restricciones diferentes?

**Responde esta pregunta en notebook.md.**


## 4. Qué es un DAG

Un **DAG** es un *Directed Acyclic Graph*: un grafo dirigido acíclico. “Dirigido” conserva la precedencia de las flechas; “acíclico” significa que no existe un camino dirigido que regrese al punto de partida.

```text
DAG:          A → C → D       con B → C
con ciclo:    A → B → C → A
```

El primer grafo admite órdenes como `A, B, C, D` o `B, A, C, D`. El segundo exige simultáneamente A antes que B, B antes que C y C antes que A: ninguna secuencia lineal satisface todo.

La propiedad clave es: un grafo dirigido tiene orden topológico **si y solo si** no contiene ciclos. Aquí no demostraremos formalmente ambos sentidos; lo observaremos mediante nodos disponibles y cantidad procesada.

**Actividad.** Clasifica `X → Y`, dos nodos aislados, una autoarista `Z → Z` y `P → Q → R → P`. Resultado esperado: los dos primeros son DAG; los dos últimos contienen ciclo.

<div style="border-left:6px solid #315f8c;background:#eef5fb;color:#17212b;padding:14px 18px;margin:14px 0"><strong>NOTA</strong><br>Un grafo desconectado puede ser DAG. La conectividad no es requisito para ordenar dependencias.</div>

Para descubrir qué nodo puede salir primero, necesitamos contar aristas entrantes.

### Comprueba tu comprensión

**Pregunta:** ¿Por qué el ciclo A → B → C → A impide cualquier orden topológico?

**Responde esta pregunta en notebook.md.**


## 5. Ejemplos con y sin ciclos

Un dibujo puede ocultar ciclos largos, así que conviene traducirlos a restricciones. Considera cuatro casos:

| Caso | Aristas | Clasificación |
| --- | --- | --- |
| cadena | A→B, B→C | DAG, orden único |
| fuentes | A→C, B→C | DAG, varios órdenes |
| ciclo | A→B, B→A | sin orden |
| mixto | D→E y A→B→C→A | el grafo completo no es DAG |

En el caso mixto sí podemos procesar D y E, pero eso no rescata al grafo completo: A, B y C seguirán bloqueados. Este ejemplo será importante cuando detectemos ciclos contando nodos procesados.

**Desarrollo manual.** Escribe una secuencia candidata para cada DAG y marca la primera arista que invalida cada propuesta incorrecta. No basta decir “se ve mal”; localiza `u → v` con `v` antes que `u`.

<div style="border-left:6px solid #b26a00;background:#fff6e6;color:#3d2b10;padding:14px 18px;margin:14px 0"><strong>ERROR FRECUENTE</strong><br>Procesar una componente acíclica no demuestra que todo el grafo sea acíclico.</div>

La siguiente sección convertirá estas restricciones en una tabla numérica.

### Comprueba tu comprensión

**Pregunta:** En el grafo mixto, ¿por qué procesar D y E no permite devolver un orden parcial como solución?

**Responde esta pregunta en notebook.md.**


## 6. Grado de entrada

El **grado de entrada** de un nodo es la cantidad de aristas que llegan a él. En dependencias representa cuántos requisitos directos siguen pendientes al inicio.

```text
A → C
B → C
C → D
```

| Nodo | Aristas entrantes | Grado |
| --- | --- | ---: |
| A | ninguna | 0 |
| B | ninguna | 0 |
| C | A→C, B→C | 2 |
| D | C→D | 1 |

**Actividad 1 — calcular grados.** Completa la misma tabla para el plan de cursos. Los primeros resultados son: Programación 0 y Estructuras de Datos 2. Comprueba que la suma de todos los grados coincide con la cantidad de dependencias normalizadas.

Contrato de código: `grados_entrada(grafo)` devuelve una entrada para cada nodo explícito, vecino implícito y aislado; no modifica el grafo. Una dependencia repetida se cuenta una sola vez después de normalizar.

<div style="border-left:6px solid #b42318;background:#fff0ef;color:#341313;padding:14px 18px;margin:14px 0"><strong>PRECAUCIÓN</strong><br>La longitud de la lista de vecinos es grado de salida, no grado de entrada.</div>

Ya podemos reconocer qué tareas no esperan a nadie.

### Comprueba tu comprensión

**Pregunta:** ¿Qué grados de entrada tienen Algoritmos, Optimización y Proyecto Final en el problema conductor?

**Responde esta pregunta en notebook.md.**


## 7. Nodos disponibles

Un nodo con grado de entrada **actual** cero no tiene requisitos pendientes y puede procesarse sin violar dependencias. En el plan inicial están disponibles Programación, Matemáticas Discretas y Álgebra.

La palabra “actual” importa. Estructuras de Datos inicia con grado 2. Después de procesar Programación pasa a 1; después de Matemáticas Discretas pasa a 0 y recién entonces entra en la cola.

```text
grado inicial de Estructuras de Datos: 2
procesar Programación:                 1
procesar Matemáticas Discretas:        0 → disponible
```

**Comprobación.** Un estado que muestre a un nodo de grado 2 dentro de la cola es inválido. Un nodo aislado, en cambio, tiene grado cero y debe aparecer en algún lugar del resultado.

<div style="border-left:6px solid #315f8c;background:#eef5fb;color:#17212b;padding:14px 18px;margin:14px 0"><strong>INVARIANTE CENTRAL</strong><br>Todo nodo que está en la cola tiene grado de entrada actual cero.</div>

Con esta regla ya podemos descubrir el algoritmo completo en vez de memorizar una receta.

### Comprueba tu comprensión

**Pregunta:** ¿En qué momento exacto debe encolarse Estructuras de Datos?

**Responde esta pregunta en notebook.md.**


## 8. Descubrimiento de Kahn

El algoritmo de Kahn surge al repetir esta pregunta: **¿qué tarea podemos ejecutar ahora?** Primero normalizamos el grafo, calculamos grados y encolamos todas las fuentes. Después procesamos una, la agregamos al orden y reducimos los grados de sus sucesores.

```text
normalizar
calcular grados
encolar grados cero
mientras haya disponibles:
    desencolar actual
    agregarlo al orden
    por cada vecino:
        reducir su grado
        si llega a cero, encolarlo
comprobar cuántos nodos se procesaron
```

No borramos físicamente aristas: la reducción en una copia de grados representa que una dependencia quedó satisfecha. Así conservamos el grafo original para pruebas, validación o reutilización.

**Actividad.** Antes de ejecutar, predice cola inicial, primer nodo y qué grados cambiarán para `A→C`, `B→C`, `C→D`. Resultado verificable: C entra únicamente después de A y B.

<div style="border-left:6px solid #2f7d5b;background:#edf8f2;color:#173b2b;padding:14px 18px;margin:14px 0"><strong>CONSEJO</strong><br>Separa “estar disponible”, “estar en la cola” y “ya procesado”; son estados distintos.</div>

Ahora realizaremos la ejecución completa sin ocultar ninguna actualización.

### Comprueba tu comprensión

**Pregunta:** ¿Qué representa disminuir en uno el grado de entrada de un vecino?

**Responde esta pregunta en notebook.md.**


## 9. Ejecución manual

Ejecuta `A→C`, `B→C`, `C→D`. Grados iniciales: A=0, B=0, C=2, D=1; cola inicial `A → B`.

| Paso | Cola antes | Actual | Vecino | Grado anterior | Grado nuevo | Acción | Orden |
| ---: | --- | --- | --- | ---: | ---: | --- | --- |
| 1 | A, B | A | C | 2 | 1 | sigue bloqueado | A |
| 2 | B | B | C | 1 | 0 | encolar C | A, B |
| 3 | C | C | D | ? | ? | ? | ? |
| 4 | ? | D | — | — | — | terminar | ? |

**Actividad 2 — completar Kahn.** Llena las incógnitas y repite el ejercicio comenzando con B. Debes obtener dos órdenes válidos: `A,B,C,D` y `B,A,C,D`.

Usa después el visualizador. Antes de **Siguiente**, predice nodo actual, grado que cambiará y contenido de la cola. La comprobación no consiste en copiar el frame, sino en comparar tu hipótesis.

<div style="border-left:6px solid #7b4bb7;background:#f4effa;color:#2f2141;padding:14px 18px;margin:14px 0"><strong>ACTIVIDAD VISUAL</strong><br>Explora cadena, varias fuentes, órdenes múltiples, ciclos, aislados, duplicados y el plan de cursos.</div>

La ejecución usa una cola; la siguiente sección explica por qué `deque` es suficiente y cómo se conecta con Clase 17.

### Comprueba tu comprensión

**Pregunta:** Completa los pasos 3 y 4: ¿qué valores y orden final deben aparecer?

**Responde esta pregunta en notebook.md.**


In [ ]:
from pathlib import Path
import sys
bases=[Path.cwd(),*Path.cwd().parents]
candidatas=[]
for base in bases:
    candidatas.extend([base,base/'clase_19',base/'curso-alumnos'/'clase_19'])
RAIZ_CLASE=next((r for r in candidatas if (r/'src'/'visualizador_topologico.py').exists()),None)
if RAIZ_CLASE is None:
    raise FileNotFoundError('No se encontró curso-alumnos/clase_19')
sys.path.insert(0,str(RAIZ_CLASE))
from src.visualizador_topologico import diagnosticar_widgets, mostrar_topologico
print(diagnosticar_widgets())
mostrar_topologico()


## 10. Uso de la cola

La implementación principal usa `collections.deque`: `append` encola y `popleft` desencola, ambos en O(1). Esto concentra la práctica en grados y dependencias, sin convertir un error previo de enlaces en un bloqueo nuevo.

```python
disponibles = deque(nodo for nodo in grafo if grados[nodo] == 0)
actual = disponibles.popleft()
disponibles.append(vecino)
```

La misma política puede implementarse con `ColaLigada` de la Clase 17: `encolar`, `desencolar` y `esta_vacia`. No copiamos su implementación al notebook; reutilizamos su contrato. La evaluación de esta clase no depende de que la cola propia esté terminada.

La cola conserva el orden de aparición entre fuentes, pero ese detalle no convierte una salida en “la única”. Si se pide el orden lexicográficamente mínimo, la operación cambia y usaríamos un min-heap.

<div style="border-left:6px solid #315f8c;background:#eef5fb;color:#17212b;padding:14px 18px;margin:14px 0"><strong>NOTA</strong><br>La estructura auxiliar realiza la política; no define por sí misma el objetivo del algoritmo.</div>

Ya conocemos la herramienta; ahora fijaremos las propiedades que deben sobrevivir cada iteración.

### Comprueba tu comprensión

**Pregunta:** ¿Por qué la solución evaluada usa deque aunque ColaLigada pueda implementar la misma política?

**Responde esta pregunta en notebook.md.**


## 11. Invariantes

Los invariantes permiten revisar un estado sin repetir toda la ejecución:

1. Todo nodo en cola tiene grado actual cero.
2. Ningún nodo se procesa dos veces.
3. El orden contiene solo nodos ya procesados.
4. Cada reducción representa una arista satisfecha.
5. Un vecino se encola justo cuando su grado llega a cero.
6. Ningún grado se vuelve negativo.
7. El grafo original permanece igual.

**Actividad de diagnóstico.** Clasifica estos estados: (a) B está en cola con grado 2; (b) C aparece dos veces en el orden; (c) una dependencia duplicada provoca grado −1; (d) un aislado está en la cola inicial. Los tres primeros violan invariantes; el cuarto es correcto.

No se necesita un conjunto explícito de visitados si normalización, grados y regla de encolado son correctos: cada grado cruza a cero una sola vez. Agregar `visitados` puede ocultar un error en vez de arreglarlo.

<div style="border-left:6px solid #b42318;background:#fff0ef;color:#341313;padding:14px 18px;margin:14px 0"><strong>ERROR FRECUENTE</strong><br>Usar visitados por imitación de BFS sin comprender qué evita realmente los duplicados.</div>

Los invariantes serán el mapa para leer la implementación paso a paso.

### Comprueba tu comprensión

**Pregunta:** ¿Qué invariante se viola si un nodo entra a la cola cuando su grado todavía es 1?

**Responde esta pregunta en notebook.md.**


## 12. Implementación paso a paso

`orden_topologico` separa responsabilidades. Primero obtiene una copia normalizada; después calcula grados; luego crea una cola y mantiene un orden. El bucle no modifica listas de vecinos.

```python
def orden_topologico(grafo):
    normalizado = normalizar_grafo_dirigido(grafo)
    grados = ...
    disponibles = deque(...)
    orden = []
    while disponibles:
        actual = disponibles.popleft()
        orden.append(actual)
        for vecino in normalizado[actual]:
            grados[vecino] -= 1
            if grados[vecino] == 0:
                disponibles.append(vecino)
    ...
```

**Contrato:** vacío devuelve `[]`; aislado aparece una vez; DAG devuelve todos los nodos; ciclo devuelve `None`; no se muta la entrada. La base contiene firmas y docstrings, pero tú completas cada responsabilidad.

**Estrategia.** Implementa y prueba en este orden: normalización, grados, validación de órdenes y finalmente Kahn. Así un fallo se localiza en una capa concreta.

<div style="border-left:6px solid #b26a00;background:#fff6e6;color:#3d2b10;padding:14px 18px;margin:14px 0"><strong>ADVERTENCIA</strong><br>No ordenes ni elimines elementos de las listas recibidas; crea estructuras nuevas.</div>

Falta decidir qué devolver cuando el bucle se detiene antes de cubrir el grafo.

### Comprueba tu comprensión

**Pregunta:** ¿Qué cuatro estructuras locales necesita mantener orden_topologico y para qué sirve cada una?

**Responde esta pregunta en notebook.md.**


## 13. Detección de ciclos

Para `A→B→C→A`, todos los grados iniciales son 1. La cola empieza vacía y no se procesa nada. En el grafo mixto `D→E` más ese ciclo, sí se procesan D y E, pero quedan A, B y C.

```python
if len(orden) != len(normalizado):
    return None
```

La desigualdad detecta que alguna componente no pudo liberar sus dependencias. No necesitamos localizar el ciclo exacto para cumplir este contrato.

**Actividad 3 — detectar ciclos.** Clasifica: cadena; dos fuentes hacia un sumidero; autoarista; componente acíclica junto a un ciclo. Para cada caso anota cola inicial, cantidad procesada y retorno esperado.

Resultado: una autoarista nunca llega a grado cero; el caso mixto procesa solo su parte acíclica y aun así devuelve `None`.

<div style="border-left:6px solid #315f8c;background:#eef5fb;color:#17212b;padding:14px 18px;margin:14px 0"><strong>IMPORTANTE</strong><br>Que la cola quede vacía es normal al terminar un DAG. El ciclo se decide comparando la cantidad procesada.</div>

Como Kahn usa cola, ahora debemos evitar confundirlo con BFS.

### Comprueba tu comprensión

**Pregunta:** ¿Por qué len(orden) != len(normalizado) es evidencia de un ciclo?

**Responde esta pregunta en notebook.md.**


## 14. BFS frente a Kahn

Ambos algoritmos usan cola, pero responden preguntas distintas.

| Característica | BFS | Kahn |
| --- | --- | --- |
| cola inicial | un origen | todos los grados cero |
| regla de entrada | vecino no descubierto | grado llega a cero |
| estado | visitados/distancias | grados pendientes |
| objetivo | recorrer o hallar caminos | ordenar dependencias |
| ciclos | no impiden recorrer | impiden ordenar todo |

En BFS, B puede descubrirse desde un solo predecesor aunque existan otras aristas entrantes. En Kahn, B espera hasta que **todas** sus dependencias se hayan procesado. Esa diferencia explica por qué copiar “visitados” y un origen produce un algoritmo incorrecto.

**Comprobación.** Para `A→C` y `B→C`, un BFS desde A puede visitar C sin procesar B. Kahn no debe hacerlo: C conserva grado 1.

<div style="border-left:6px solid #7b4bb7;background:#f4effa;color:#2f2141;padding:14px 18px;margin:14px 0"><strong>CONCLUSIÓN</strong><br>Compartir estructura auxiliar no implica compartir estado, regla ni objetivo.</div>

Esta diferencia también explica por qué puede haber varios órdenes correctos.

### Comprueba tu comprensión

**Pregunta:** ¿Cuál es la diferencia decisiva entre la regla para encolar en BFS y en Kahn?

**Responde esta pregunta en notebook.md.**


## 15. Órdenes no únicos

Si varios nodos tienen grado cero, cualquiera puede procesarse primero. Para `A→C` y `B→C`, `A,B,C` y `B,A,C` son válidos; `C,A,B` no lo es.

Esto cambia el diseño de pruebas. Comparar únicamente con una lista esperada rechazaría soluciones correctas. Una prueba por propiedades debe verificar:

- aparecen todos los nodos;
- no hay duplicados;
- no aparecen desconocidos;
- para cada `u→v`, la posición de u es menor que la de v.

La implementación con `deque` conserva el orden de primera aparición entre disponibles y por eso es determinista para una entrada concreta, pero el contrato permite otras salidas.

**Actividad.** Enumera todos los órdenes válidos de `A→D`, `B→D`, `C→D`. Resultado: cualquier permutación de A, B y C seguida de D; son seis.

<div style="border-left:6px solid #b26a00;background:#fff6e6;color:#3d2b10;padding:14px 18px;margin:14px 0"><strong>ERROR DE PRUEBA</strong><br>No confundas determinismo de una implementación con unicidad matemática del orden.</div>

Necesitamos entonces una función que valide secuencias por propiedades.

### Comprueba tu comprensión

**Pregunta:** ¿Por qué un test que exige exactamente [A, B, C] es incorrecto para A→C y B→C?

**Responde esta pregunta en notebook.md.**


## 16. Validación de un orden

`es_orden_topologico(grafo, orden)` construye posiciones y revisa todas las aristas. Antes comprueba longitud, nodos conocidos, cobertura y ausencia de repetidos.

```python
posicion = {nodo: indice for indice, nodo in enumerate(orden)}
# para cada u → v debe cumplirse posicion[u] < posicion[v]
```

**Actividad 4 — validar secuencias.** Para `A→C`, `B→C`, clasifica:

| Secuencia | ¿Válida? | Primera razón |
| --- | --- | --- |
| A,B,C | sí | respeta ambas aristas |
| B,A,C | ? | ? |
| C,A,B | ? | ? |
| A,C | ? | ? |
| A,A,C | ? | ? |

El validador no ejecuta Kahn nuevamente: verifica el producto. Esto permite probar cualquier implementación y documentar contraejemplos precisos.

<div style="border-left:6px solid #315f8c;background:#eef5fb;color:#17212b;padding:14px 18px;margin:14px 0"><strong>CONTRATO</strong><br>Longitud correcta no basta: una secuencia puede repetir A y omitir B.</div>

Antes de calcular grados o validar, debemos garantizar una representación coherente.

### Comprueba tu comprensión

**Pregunta:** Clasifica las cuatro secuencias incompletas de la tabla y explica la primera regla que falla.

**Responde esta pregunta en notebook.md.**


## 17. Normalización y dependencias duplicadas

La entrada puede usar listas o tuplas, mencionar destinos sin clave propia y repetir dependencias. `normalizar_grafo_dirigido` crea una copia estable:

```python
{"A": ["B", "B", "C"]}
# → {"A": ["B", "C"], "B": [], "C": []}
```

Cada nodo y vecino debe ser `str`; una adyacencia no puede ser un string disfrazado de secuencia. Las listas de salida son nuevas y no se comparten con la entrada. Los vecinos implícitos se agregan como nodos sin sucesores.

Eliminar duplicados es parte del contrato algorítmico: si A→B se contara dos veces pero al procesar A solo se redujera una, B quedaría bloqueado; si se redujera de más, aparecerían grados negativos. Normalizar una sola vez evita ambas inconsistencias.

**Actividad.** Predice claves y listas para `{"A": ("C","C"), "B": []}`. Resultado según primera aparición: A, C, B; C aparece una vez en la adyacencia.

<div style="border-left:6px solid #b42318;background:#fff0ef;color:#341313;padding:14px 18px;margin:14px 0"><strong>PRECAUCIÓN</strong><br>No uses <code>set</code> como salida: eliminaría duplicados, pero perdería el orden estable y el tipo esperado.</div>

Una representación robusta permite tratar casos límite sin excepciones improvisadas.

### Comprueba tu comprensión

**Pregunta:** ¿Por qué una dependencia duplicada no debe aumentar dos veces el grado de entrada?

**Responde esta pregunta en notebook.md.**


## 18. Casos límite

Los casos pequeños revelan el contrato:

| Entrada | Resultado esperado |
| --- | --- |
| `{}` | `[]` |
| `{"A": []}` | `["A"]` |
| `{"A": ["A"]}` | `None` |
| `{"A": ["B"]}` | A antes que B; B implícito |
| `{"A": [], "Z": []}` | ambos aparecen |

Un grafo desconectado acíclico sigue siendo ordenable. Un ciclo en cualquier componente invalida el resultado completo. El grafo original debe conservar sus listas, duplicados y orden tal como fue recibido, aunque la copia normalizada cambie la representación interna.

**Comprobación de no mutación.** Haz `copia = deepcopy(grafo)`, ejecuta las cuatro funciones y compara `grafo == copia`. También comprueba que modificar una lista normalizada no altera la original.

<div style="border-left:6px solid #2f7d5b;background:#edf8f2;color:#173b2b;padding:14px 18px;margin:14px 0"><strong>CONSEJO DE PRUEBA</strong><br>Una función puede devolver lo correcto y aun incumplir el contrato si modifica su entrada.</div>

Con los contratos generales listos, aplicaremos Kahn a cursos numerados.

### Comprueba tu comprensión

**Pregunta:** ¿Cómo deben aparecer los nodos aislados en un orden topológico y por qué?

**Responde esta pregunta en notebook.md.**


## 19. Problema de cursos

`ordenar_cursos(numero_cursos, prerrequisitos)` usa nodos enteros `0..n−1`. En esta clase cada par es `(prerrequisito, curso)`, es decir, una arista hacia el curso que espera.

Contratos: cantidad entera no negativa y no booleana; pares de dos enteros no booleanos; índices dentro de rango; duplicados ignorados; auto dependencia produce ciclo; entrada sin prerrequisitos devuelve todos los cursos; la lista original no cambia.

```text
n = 4
(0, 2), (1, 2), (2, 3)
fuentes: 0, 1
órdenes válidos: 0,1,2,3 o 1,0,2,3
```

`puede_completar_cursos` reutiliza el resultado: no duplica Kahn, solo comprueba si `ordenar_cursos(...) is not None`.

**Actividad.** Añade `(3,0)` y localiza el ciclo. Después prueba una dependencia duplicada y explica por qué el resultado lógico no cambia.

<div style="border-left:6px solid #b42318;background:#fff0ef;color:#341313;padding:14px 18px;margin:14px 0"><strong>TIPOS</strong><br><code>True</code> es subtipo de <code>int</code> en Python, pero no representa una cantidad ni un índice válido en este contrato.</div>

El problema principal externo usa la misma orientación y solo requiere adaptar índices.

### Comprueba tu comprensión

**Pregunta:** ¿Qué significa exactamente el par (2, 5) en ordenar_cursos?

**Responde esta pregunta en notebook.md.**


## 20. CSES Course Schedule

El problema oficial **CSES Course Schedule** pide ordenar cursos numerados de 1 a n con requisitos `a antes que b`. Acepta cualquier orden válido y exige `IMPOSSIBLE` si no existe.

La adaptación tiene tres pasos: leer pares 1-based, restar uno para reutilizar `ordenar_cursos`, y sumar uno al imprimir. El algoritmo central no debe mezclar lógica de entrada/salida con Kahn.

```text
juez: 1 → 2
función reutilizable: (0, 1)
salida: cada curso + 1
```

**Práctica guiada.** Para el ejemplo oficial con cinco cursos, valida la salida comprobando cada requisito, no comparando con una única secuencia. En alumnos no se entrega aquí la solución completa del juez: primero implementa las funciones reutilizables.

<div style="border-left:6px solid #315f8c;background:#eef5fb;color:#17212b;padding:14px 18px;margin:14px 0"><strong>REFERENCIA</strong><br>Consulta el enlace oficial desde README o la práctica. No copies el enunciado completo en tu entrega.</div>

Dos problemas relacionados cambian ligeramente la pregunta y la convención de pares.

### Comprueba tu comprensión

**Pregunta:** ¿Qué conversiones de índices necesita la adaptación de CSES y qué debe imprimirse si hay ciclo?

**Responde esta pregunta en notebook.md.**


## 21. LeetCode Course Schedule

LeetCode 207 pregunta si todos los cursos pueden completarse; basta comprobar que Kahn procese todos. LeetCode 210 pide devolver un orden. Ambos suelen representar cada par como `[curso, prerrequisito]`, convención inversa a nuestra función.

```text
LeetCode [curso, prerrequisito]
clase     (prerrequisito, curso)
```

La adaptación correcta transforma los pares una vez y delega en `puede_completar_cursos` u `ordenar_cursos`. No reimplementa el algoritmo. En LeetCode 210 un ciclo suele expresarse con lista vacía, mientras nuestra función general devuelve `None` para distinguirlo del grafo vacío.

**Actividad.** Convierte `[[1,0],[2,0],[3,1],[3,2]]` a nuestra convención y dibuja el DAG. Hay más de un orden válido porque 1 y 2 pueden intercambiarse.

<div style="border-left:6px solid #b26a00;background:#fff6e6;color:#3d2b10;padding:14px 18px;margin:14px 0"><strong>ADVERTENCIA</strong><br>No mezcles dos convenciones dentro de la misma función; adapta en el borde y documenta.</div>

Con el algoritmo aplicado, podemos justificar su costo con el trabajo realmente realizado.

### Comprueba tu comprensión

**Pregunta:** ¿Cómo se relacionan ordenar_cursos, puede_completar_cursos y las preguntas de LeetCode 207/210?

**Responde esta pregunta en notebook.md.**


## 22. Complejidad

Normalizar visita cada nodo y dependencia: O(V+E). Calcular grados vuelve a recorrer la representación: O(V+E). Kahn encola y procesa cada nodo una vez y reduce cada arista una vez: O(V+E). Validar un orden construye posiciones O(V) y revisa aristas O(E).

```text
trabajo por nodo: inicializar, encolar como máximo una vez, procesar una vez
trabajo por arista: contar una vez, reducir una vez, validar una vez
```

Las fases son consecutivas, no multiplicativas: O(V+E)+O(V+E) sigue siendo O(V+E), no O(VE). La memoria auxiliar es O(V+E) por la copia normalizada, grados, cola y orden.

**Comprobación.** En una cadena de n nodos hay n−1 aristas y el bucle no reinicia una búsqueda global tras cada nodo. En un grafo denso E puede dominar, pero cada arista sigue examinándose una cantidad constante de veces.

<div style="border-left:6px solid #315f8c;background:#eef5fb;color:#17212b;padding:14px 18px;margin:14px 0"><strong>INTUICIÓN</strong><br>Contar bucles anidados sin preguntar qué elementos recorren produce análisis incorrectos.</div>

La complejidad debe quedar protegida indirectamente por pruebas grandes y por ausencia de recorridos innecesarios.

### Comprueba tu comprensión

**Pregunta:** ¿Por qué el bucle anidado sobre nodos y vecinos no implica O(VE)?

**Responde esta pregunta en notebook.md.**


## 23. Pruebas

Una prueba debe nombrar el contrato que protege. Casos obligatorios propios: varios órdenes; ciclo simple; ciclo parcial; vecino implícito; duplicados; orden incorrecto. Recomendados: vacío, aislado, cadena, autoarista, no mutación e índices inválidos.

Ejemplo de propiedad para órdenes no únicos:

```python
orden = orden_topologico(grafo)
assert orden is not None
assert es_orden_topologico(grafo, orden)
```

No uses la función bajo prueba como única evidencia de sí misma: en al menos una prueba pequeña calcula posiciones directamente. Para ciclos, exige `None`, no un prefijo. Para copias defensivas, compara antes/después y modifica la salida normalizada.

**Actividad.** Diseña seis pruebas y documenta regla, entrada, esperado y relevancia. Ejecuta públicas más propias con `evaluar.py` y conserva la salida completa de `pytest -v`.

<div style="border-left:6px solid #2f7d5b;background:#edf8f2;color:#173b2b;padding:14px 18px;margin:14px 0"><strong>CONSEJO</strong><br>Una prueba útil falla por una razón comprensible cuando se rompe un solo contrato.</div>

La última extensión cambia el criterio de desempate sin cambiar las dependencias.

### Comprueba tu comprensión

**Pregunta:** ¿Cómo debe probarse un resultado cuando el grafo admite varios órdenes topológicos?

**Responde esta pregunta en notebook.md.**


## 24. Extensión con heap

La versión obligatoria devuelve cualquier orden válido con una cola. Si el contrato pidiera el orden lexicográficamente mínimo, entre todos los nodos disponibles habría que extraer siempre el menor. Esa operación dominante corresponde a un min-heap.

```text
cualquier disponible válido → deque
menor disponible            → heap
```

`orden_topologico_minimo` es una extensión docente y no aparece en las pruebas públicas. Conecta con heaps de Clase 14 y con Dijkstra: el heap no convierte el algoritmo en Dijkstra; solo implementa otra política de selección.

El costo pasa típicamente a O((V+E) log V), porque inserciones y extracciones del conjunto disponible cuestan logarítmicamente. La regla de grados y la detección de ciclos permanecen iguales.

**Actividad opcional.** Para fuentes B, A y C hacia D, compara la salida FIFO según primera aparición con la salida mínima. Resultado mínimo: A,B,C,D.

<div style="border-left:6px solid #7b4bb7;background:#f4effa;color:#2f2141;padding:14px 18px;margin:14px 0"><strong>EXTENSIÓN</strong><br>No sustituyas la solución obligatoria ni agregues complejidad si el problema acepta cualquier orden.</div>

Terminaremos integrando la operación dominante de todos los algoritmos del cierre.

### Comprueba tu comprensión

**Pregunta:** ¿Qué cambio de contrato justifica sustituir la cola por un heap?

**Responde esta pregunta en notebook.md.**


## 25. Cierre integrador

La clase comenzó con tareas bloqueadas y terminó con una política verificable:

| Problema | Operación dominante | Estructura | Algoritmo |
| --- | --- | --- | --- |
| camino sin pesos | procesar por niveles | cola | BFS |
| pesos 0/1 | atender costo cero primero | deque | 0-1 BFS |
| pesos no negativos | extraer menor distancia | heap | Dijkstra |
| conectar toda la red | consultar componentes | Union-Find | Kruskal |
| ordenar dependencias | procesar sin requisitos | cola + grados | Kahn |

En Kahn, el invariante central es que todo nodo en la cola tiene grado cero. El producto es una secuencia que cubre todos los nodos y respeta cada arista. Si la cobertura falla, existe ciclo. Normalización, no mutación y validación hacen que esa idea sea reutilizable y testeable.

**Síntesis personal.** Explica el problema, la operación, la estructura, el invariante, la complejidad y un caso donde no aplica. Después compara con el algoritmo anterior que más se le parece.

<div style="border-left:6px solid #315f8c;background:#eef5fb;color:#17212b;padding:14px 18px;margin:14px 0"><strong>CLASE 20 — LABORATORIO INTEGRADOR</strong><br>No aprenderemos otro algoritmo: elegiremos la estructura adecuada a partir de un problema nuevo.</div>

La pregunta final prepara esa decisión profesional.

### Comprueba tu comprensión

**Pregunta:** Ante un problema nuevo, ¿cómo identificamos la operación dominante y elegimos la estructura de datos adecuada?

**Responde esta pregunta en notebook.md.**
